# Fine-tune DictaLM 3.0 1.7B (Unsloth LoRA) — Day 2

Runtime: **L4 or A100** (Runtime → Change runtime type). Run the sections in order; each one is a task from the guide.

## Setup
**Cell 1 — install** (once per new machine; "session crashed" at the end is expected: it restarts Python so the new packages load).

In [ ]:
!pip install -q unsloth
import os
os.kill(os.getpid(), 9)

**Cell 2 — Drive, repo, token** (run after every restart).

In [ ]:
import os, sys, json
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

REPO = "/content/bureaucracy-navigator"
if not os.path.exists(REPO):
    !git clone https://github.com/AhmadTawil1/bureaucracy-navigator.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}

DRIVE_OUT = "/content/drive/MyDrive/bureaucracy-navigator/training"
os.makedirs(DRIVE_OUT, exist_ok=True)

## Task 2.1 — Model, LoRA, data, label check

In [ ]:
from unsloth import FastLanguageModel

MAX_LEN = 4096  # longest training example is 2,839 tokens
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="dicta-il/DictaLM-3.0-1.7B-Instruct",
    max_seq_length=MAX_LEN,
    load_in_4bit=False,  # 1.7B fits in 16-bit: better than QLoRA
    token=os.environ["HF_TOKEN"],
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "data/out/train.jsonl", "validation": "data/out/val.jsonl"})
ds = ds.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)},
            remove_columns=ds["train"].column_names)
print(ds)
print(ds["train"][0]["text"][-600:])  # markers: "<|im_start|>user\n" ... "<|im_start|>assistant\n"

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_LEN,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,   # effective batch 16 → ~71 steps per epoch
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=10,
        eval_strategy="steps", eval_steps=20,
        save_strategy="steps", save_steps=20, save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",  # keeps the best checkpoint if val loss rises
        bf16=True,
        output_dir="/content/outputs",
        report_to="none",
        seed=42,
    ),
)
# Train only on the assistant's answer, not on the prompt/context.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

# Check: the labels of one example decode to ONLY the assistant's answer.
row = trainer.train_dataset[0]
print(tokenizer.decode([t for t in row["labels"] if t != -100]))

## Task 2.2 — Baseline answers (before training)

In [ ]:
val = [json.loads(line) for line in open("data/out/val.jsonl", encoding="utf-8")]
samples = [next(e for e in val if e["type"] == t) for t in ("qa", "unanswerable", "letter")]

def generate(example, max_new_tokens=600):
    prompt = tokenizer.apply_chat_template(example["messages"][:2], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # greedy: same input → same output
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
before = [generate(e) for e in samples]
FastLanguageModel.for_training(model)
json.dump(before, open(f"{DRIVE_OUT}/before.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
for e, answer in zip(samples, before):
    print(f"\n===== [{e['type']}] {e['article']}\n{answer}")

## Task 2.3 — Train (~15–25 min)

In [ ]:
stats = trainer.train()
print(stats)

## Task 2.4 — Check the training

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_loss = [(h["step"], h["loss"]) for h in log if "loss" in h]
val_loss = [(h["step"], h["eval_loss"]) for h in log if "eval_loss" in h]
plt.plot(*zip(*train_loss), label="train")
plt.plot(*zip(*val_loss), label="validation", marker="o")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()
print("best checkpoint:", trainer.state.best_model_checkpoint, "eval_loss:", trainer.state.best_metric)

In [ ]:
FastLanguageModel.for_inference(model)
after = [generate(e) for e in samples]
json.dump(after, open(f"{DRIVE_OUT}/after.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
for e, b, a in zip(samples, before, after):
    print(f"\n===== [{e['type']}] {e['article']}\n--- BEFORE:\n{b}\n--- AFTER:\n{a}")

## Task 2.5 — Save and export

In [ ]:
from huggingface_hub import whoami

LORA_DIR = f"{DRIVE_OUT}/dictalm3-bureaucracy-lora"
model.save_pretrained(LORA_DIR)       # adapters only (small)
tokenizer.save_pretrained(LORA_DIR)

HF_REPO = f"{whoami(token=os.environ['HF_TOKEN'])['name']}/dictalm3-bureaucracy-lora"
model.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"], private=True)
tokenizer.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"], private=True)
print("saved to", LORA_DIR, "and", HF_REPO)

In [ ]:
import glob, shutil

model.save_pretrained_gguf("/content/dictalm3-bureaucracy", tokenizer, quantization_method="q8_0")
ggufs = glob.glob("/content/**/*.gguf", recursive=True)
print(ggufs)
for path in ggufs:
    shutil.copy(path, DRIVE_OUT)
print("copied to", DRIVE_OUT)